# 02 — Preconditions, calibration, baseline network

Second of four (spec §9). Runs steps 0–2: preconditions (**G0 re-baselined, G2, G8**), the
per-part CWD + locked intra-name MSTs + band-cutoff **calibration** (step 1, written into
`run_config.json`), and the **baseline network** — locked + inter-name MST + β-augmentation,
criticality, bands, priority surface, v1 comparison, co-benefit audit (**G3/G4/G5**).

Creates the run dir (`output_data/corridors_north/v2_runNNN/`) that notebooks 03 and 04
re-attach to via their `RUN` variable.


In [ ]:
# ---- Setup: find the project root, import the shared engines ----------------
# This notebook lives in analyses/northern_connectivity/, below the repo root where config.py
# and the corridor engine modules (corridors_prep / corridor_graph / corridors_core /
# corridors_ensemble) sit. Same bootstrap pattern as analyses/y2y/.
import sys, pathlib
_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
          if (p / "config.py").exists()]
assert _cands, f"config.py not found above {pathlib.Path.cwd()} -- run this notebook from inside the repo"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))

import importlib
import config
import corridors_prep as cp
import corridor_graph as cg
import corridors_core as cc
import corridors_ensemble as ce
for _m in (config, cp, cg, cc, ce):
    importlib.reload(_m)

KEY = "north"


## Step 0 · Preconditions — **G2 / G8** re-assert, then the H7-gated run creation

`cc.resolve` is G8 (raises on any retired v1 key AND on any missing addendum key);
`cp.check` re-asserts warp fidelity on current disk state. `cc.start` then refuses to proceed
unless `multipart_review.csv` is SIGNED — it copies the H7 artifacts into the run dir and pins
their sha256. H1 (O'Brien licence sign-off) does NOT block the run; it blocks external release.


In [ ]:
_ = cc.resolve(KEY, require_cutoff=False)          # G8
cp.check(cp.grid(KEY))                              # G2 re-assert


In [ ]:
A = cc.start(KEY, label="v2 baseline", require_cutoff=False)
A


### gate **G0** — re-baselined (D16): name set + dedupe merges + part count + review hash


In [ ]:
cc.gate_g0(A, expect_names=42, expect_merges=3)


## Resistance · Phase 1.3 diagnostics


In [ ]:
cc.resistance(A)
cc.resistance_report(A, v1_path=config.RESULTS_DIR / "corridors_north" / "_v1_frozen" / "resistance.tif")


## Step 1 · Cost-weighted distance — per SEED PART · the expensive stage

One MCP pass per part, cached by (resistance, part-structure) hash. Multi-part units get a
derived min-field (= multi-seed CWD from the part union — the D16 field semantics). Inter-name
distance between two multipart names = min over part pairs (standard multi-seed semantics).


In [ ]:
cc.cost_distances(A)


### Calibrate the absolute band cutoff (D6) — inter-name MST only, target 18,188 km²

`cc.set_cutoff` writes the calibrated value into THIS run's `run_config.json` (the engine's only
input after start), and reports the residual. Mirror it into
`config.CORRIDORS["north"]["cwd_cutoff_abs"]` manually if future runs should inherit it.


In [ ]:
cutoff, area = cc.calibrate_cutoff(A)
cc.set_cutoff(A, cutoff, area)


## Step 2 · Network — locked intra-name MSTs + inter-name MST + bridge backup (D7/D16) · **gate G3**

Locked edges carry `edge_class="intra_name"`: real corridor land, included in criticality and
failure enumeration, band area reported as its own line (never folded into MST or augmentation
area). **G4** here reads as "β = 0 reproduces locked + inter-name MST" — locked edges are
appended regardless of β, so it holds by construction (and `cg.selftest` covers the β=0 MST).


In [ ]:
cc.corridor_network(A)


### Criticality table — two irreplaceability senses, never merged

`irreplaceable` is the D7 β-ceiling flag (no alternative LINK). The D12 `route_irreplaceable`
flag (no alternative routing WITHIN a link) is added by notebook 04.


In [ ]:
cols = ["label_i", "label_j", "edge_class", "cost", "in_mst", "is_adjacency", "ecfb_raw",
        "disconnects", "n_pairs_lost", "cost_inflation", "backup_ratio", "irreplaceable",
        "band_km2"]
A.edges[cols].sort_values(["irreplaceable", "n_pairs_lost"], ascending=False).head(20)


## Linkage priority surface (D9) + maps

Note the flagged OPEN METHODS QUESTION: locked intra-name edges have undefined quotient-graph
centrality (like adjacencies), so their land shows in `corridors.tif` and the near-optimality
surface but contributes nothing to `linkage_priority.tif`.


In [ ]:
cc.priority_surface(A)


In [ ]:
cc.map(A)


## Compare against v1


In [ ]:
j = cc.compare(A, config.RESULTS_DIR / "corridors_north" / "_v1_frozen",
               label_a="v2 (O'Brien cost, 300 m)", label_b="v1 (blend, 1 km)")


## Co-benefit audit · **gate G5** — 300 m masks crossed to the 1 km audit grid


In [ ]:
cc.corridor_profile(A, n_groups=10)


In [ ]:
cc.corridor_group_map(A)


### gate G5 — audit invariance (v1's IPCA / PA profile rows reproduce)


In [ ]:
import pandas as pd, numpy as np
old = pd.read_csv(config.RESULTS_DIR / "corridors_north" / "_v1_frozen" / "corridor_profile.csv")
new = A.profile["table"]
for area in ("proposed IPCAs", "existing PAs"):
    o = old[old.area == area].iloc[0]; n = new[new.area == area].iloc[0]
    cols = [c for c in new.columns if "richness" in c and c in old.columns]
    d = max(abs(float(o[c]) - float(n[c])) for c in cols)
    print(f"  {area:16s} max |Δrichness| over {len(cols)} axes = {d:.4f}")
    assert d < 0.02, f"G5 FAILED on {area}: the audit path changed (max Δ {d:.4f})"
print("G5 OK — the audit is unchanged by the routing-grid switch")


## Write the core outputs

`corridor_summary.json` is also the run's "done" marker in `runs.csv`. Notebook 04's
`cc.finish` re-writes it after the D11/D12 products add their columns.


In [ ]:
cc.write_run(A)
